# Neo4j 카드 샘플 점검
기존 공유 Neo4j에서 카드 샘플을 점검합니다. `graphrag-neo4j`는 다른 프로젝트 데이터와 공유되므로 `Fp` 라벨과 `FP_` 관계만 사용합니다. `WRITE=False`가 기본값이며 로컬이 아니면 쓰지 않습니다. 확인 후 출력 지우기(Clear All Outputs) 후 저장하세요. PDF 원문은 표시하지 마세요.

# 경로와 옵션
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd()
ENV_PATH = REPO_ROOT / '.env'
SAMPLE_FILES = [Path('/tmp/fp-sample-hana.cypher'), Path('/tmp/fp-sample-woori.cypher')]
node_labels = ['FpIssuer', 'FpCard', 'FpBenefit', 'FpCondition', 'FpLimit', 'FpTarget', 'FpEvidence']
WRITE = False
ALLOW_REMOTE = False
CONFIRM_DELETE = False
TOP_CARD_ID = None
static_check_ok = False
statements_by_file = {}


# 연결
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
import os
from urllib.parse import urlparse
from dotenv import load_dotenv
from neo4j import GraphDatabase
import pandas as pd
load_dotenv(ENV_PATH)
_required = ['NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD', 'NEO4J_DATABASE']
_missing = [key for key in _required if not os.environ.get(key)]
driver = None
connected = False
is_local = False
if _missing:
    print('누락 변수 이름:', ', '.join(_missing))
else:
    _uri = os.environ['NEO4J_URI']
    _host = (urlparse(_uri).hostname or '').lower()
    is_local = _host in {'localhost', '127.0.0.1', '::1', 'host.docker.internal'}
    print('호스트 종류:', '로컬' if is_local else '원격(neo4j.io 포함)')
    if not is_local and not ALLOW_REMOTE:
        print('WRITE 불가: 원격 연결은 ALLOW_REMOTE=True 필요')
    try:
        driver = GraphDatabase.driver(_uri, auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD']))
        driver.verify_connectivity()
        connected = True
        print('연결 성공')
    except Exception:
        print('연결 실패')


# 기존 Fp 라벨 점검
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
labels_df = pd.DataFrame()
if connected:
    with driver.session(database=os.environ['NEO4J_DATABASE']) as session:
        labels_df = pd.DataFrame([record.data() for record in session.run('CALL db.labels()')])
display(labels_df)
has_fp_labels = not labels_df.empty and labels_df.iloc[:, 0].astype(str).str.startswith('Fp').any()
print('Fp 라벨 존재:', bool(has_fp_labels))
if has_fp_labels:
    print('기존 Fp 데이터가 같은 샘플의 재적재인지 먼저 확인하세요.')


# 샘플 파일 정적 검사
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
import re

allowed_relationships = {
    'FP_HAS_CONDITION', 'FP_HAS_LIMIT', 'FP_APPLIES_TO', 'FP_EXCLUDES',
    'FP_MUTUALLY_EXCLUSIVE', 'FP_OFFERS',
}
statements_by_file = {}
checks = []
for path in SAMPLE_FILES:
    text = path.read_text(encoding='utf-8') if path.exists() else ''
    statements = []
    current = []
    for line in text.splitlines():
        current.append(line)
        if line.rstrip().endswith(';'):
            statements.append('\n'.join(current).strip())
            current = []
    if ''.join(current).strip():
        statements.append('\n'.join(current).strip())
    statements_by_file[path] = statements

    cypher = re.sub(r'(?m)//.*$', '', text)
    forbidden = [
        name for pattern, name in [
            (r'(?i)\bDELETE\b', 'DELETE'),
            (r'(?i)\bDROP\b', 'DROP'),
            (r'(?i)\bREMOVE\b', 'REMOVE'),
            (r'(?i)\bCALL\b', 'CALL'),
            (r'(?i)\bAPOC\b', 'apoc'),
            (r'(?i)\bLOAD\s+CSV\b', 'LOAD CSV'),
            (r'(?i)\b(?:OPTIONAL\s+)?MATCH\s*\(\s*(?:[A-Za-z_]\w*\s*)?(?:\{[^}]*\}\s*)?\)', '라벨 없는 MATCH'),
        ] if re.search(pattern, cypher)
    ]

    used_labels = set()
    for node_pattern in re.findall(r'\(([^()]*)\)', cypher):
        header = node_pattern.split('{', 1)[0]
        if ':' in header:
            used_labels.update(re.findall(r':\s*([A-Za-z_]\w*)', header))
    used_relationships = set()
    untyped_relationship = False
    for relationship_pattern in re.findall(r'\[([^\[\]]*)\]', cypher):
        header = relationship_pattern.split('{', 1)[0]
        if ':' in header:
            used_relationships.update(re.findall(r':\s*([A-Za-z_]\w*)', header))
        else:
            untyped_relationship = True
    other_labels = sorted(used_labels - set(node_labels))
    other_relationships = sorted(used_relationships - allowed_relationships)
    if other_labels:
        forbidden.append('비허용 라벨: ' + ', '.join(other_labels))
    if other_relationships:
        forbidden.append('비허용 관계: ' + ', '.join(other_relationships))
    if untyped_relationship:
        forbidden.append('유형 없는 관계')
    if not path.exists():
        forbidden.append('파일 없음')
    if not statements:
        forbidden.append('문장 없음')

    checks.append({
        '파일': path.name,
        '문장 수': len(statements),
        '라벨': ', '.join(sorted(used_labels)),
        '관계': ', '.join(sorted(used_relationships)),
        '금지 패턴': ', '.join(forbidden) or '없음',
    })

inspection_df = pd.DataFrame(checks, columns=['파일', '문장 수', '라벨', '관계', '금지 패턴'])
display(inspection_df)
static_check_ok = bool(checks) and not inspection_df['금지 패턴'].ne('없음').any()


# 샘플 적재
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if not WRITE:
    print('건너뜀')
elif not connected or (not is_local and not ALLOW_REMOTE) or not static_check_ok:
    print('적재 차단: 연결·호스트·정적 검사를 확인하세요')
else:
    import time
    started = time.perf_counter()
    executed = 0
    failures = 0
    with driver.session(database=os.environ['NEO4J_DATABASE']) as session:
        for path, statements in statements_by_file.items():
            for statement in statements:
                try:
                    session.run(statement).consume()
                    executed += 1
                except Exception:
                    failures += 1
                    print('문장 실행 실패:', path.name)
                    break
            if failures:
                break
    print(f'실행 문장 수: {executed}, 실패 문장 수: {failures}, 소요 시간: {time.perf_counter()-started:.2f}초')


# Fp 그래프 확인 질의
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if connected:
    with driver.session(database=os.environ['NEO4J_DATABASE']) as session:
        def query_df(query, **params):
            return pd.DataFrame([record.data() for record in session.run(query, **params)])

        def count_nodes(where=''):
            return pd.concat([
                query_df(f"MATCH (n:{label}){where} RETURN '{label}' AS label, count(n) AS count")
                for label in node_labels
            ], ignore_index=True)

        node_counts = count_nodes()
        rel_counts = query_df(
            "MATCH (a)-[r]->(b) "
            "WHERE type(r) STARTS WITH 'FP_' "
            "AND any(label IN labels(a) WHERE label IN $labels) "
            "AND any(label IN labels(b) WHERE label IN $labels) "
            "RETURN type(r) AS relationship, count(r) AS count ORDER BY relationship",
            labels=node_labels,
        )
        cards_benefits = query_df("MATCH (c:FpCard)-[:FP_OFFERS]->(b:FpBenefit) RETURN c.name AS card, count(DISTINCT b) AS benefits ORDER BY benefits DESC")
        duplicate_benefits = query_df("MATCH (b:FpBenefit) WITH b.name AS name, collect(b.id) AS ids, count(*) AS copies WHERE copies > 1 RETURN name, ids, copies ORDER BY copies DESC LIMIT 20")
        multi_card_benefits = query_df("MATCH (c:FpCard)-[:FP_OFFERS]->(b:FpBenefit) WITH b, count(DISTINCT c) AS cards WHERE cards > 1 RETURN b.id AS id, b.name AS name, cards ORDER BY cards DESC LIMIT 10")
        evidence_status = query_df("MATCH (e:FpEvidence) RETURN e.quote_status AS quote_status, count(e) AS count")
        missing_names = count_nodes(" WHERE n.name IS NULL OR n.name = ''")

    for title, frame in [
        ('Fp 라벨별 노드 수', node_counts), ('FP_ 관계별 수', rel_counts),
        ('카드별 혜택 수', cards_benefits), ('동일 이름 혜택 상위 20', duplicate_benefits),
        ('여러 카드 연결 혜택 상위 10', multi_card_benefits),
        ('근거 quote_status별 수', evidence_status), ('name 누락 노드 수', missing_names),
    ]:
        print(title)
        display(frame)


# 최대 2단계 그래프 그림
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import font_manager

if connected:
    with driver.session(database=os.environ['NEO4J_DATABASE']) as session:
        card = session.run(
            "MATCH (c:FpCard)-[:FP_OFFERS]->(b:FpBenefit) "
            "RETURN c.id AS id, count(DISTINCT b) AS n ORDER BY n DESC, id LIMIT 1"
        ).single() if TOP_CARD_ID is None else {'id': TOP_CARD_ID}
        graph_rows = list(session.run(
            "MATCH (c:FpCard {id:$id})-[r1:FP_OFFERS]-(b:FpBenefit) "
            "OPTIONAL MATCH (b)-[r2:FP_HAS_CONDITION|FP_HAS_LIMIT|FP_APPLIES_TO|FP_EXCLUDES]-(d:FpCondition|FpLimit|FpTarget) "
            "RETURN c,b,d,r1,r2 LIMIT 80",
            id=card['id'],
        )) if card else []

    graph = nx.Graph()
    for row in graph_rows:
        for node in (row['c'], row['b'], row['d']):
            if node is not None:
                label = next(label for label in node.labels if label in node_labels)
                graph.add_node(node['id'], label=label, name=node.get('name') or node['id'])
        for relationship in (row['r1'], row['r2']):
            if relationship is not None:
                graph.add_edge(
                    relationship.start_node['id'], relationship.end_node['id'],
                    label=relationship.type,
                )

    candidates = ['Noto Sans CJK KR', 'NanumGothic', 'Malgun Gothic']
    installed = {font.name for font in font_manager.fontManager.ttflist}
    font = next((name for name in candidates if name in installed), None)
    if font:
        plt.rcParams['font.family'] = font

    colors = {
        'FpCard': '#4C78A8', 'FpBenefit': '#F58518', 'FpCondition': '#54A24B',
        'FpLimit': '#E45756', 'FpTarget': '#B279A2', 'FpIssuer': '#72B7B2',
        'FpEvidence': '#FF9DA6',
    }
    pos = nx.spring_layout(graph, seed=7)
    for label, color in colors.items():
        nodes = [node for node, data in graph.nodes(data=True) if data['label'] == label]
        if nodes:
            nx.draw_networkx_nodes(graph, pos, nodelist=nodes, node_color=color, label=label, node_size=900)
    labels = {
        node: data['name'][:12] if font else f"{data['label'][-4:]}:{node[-6:]}"
        for node, data in graph.nodes(data=True)
    }
    nx.draw_networkx_edges(graph, pos, alpha=.45)
    nx.draw_networkx_labels(graph, pos, labels, font_size=8)
    plt.legend()
    plt.axis('off')
    plt.show()


# Fp 데이터 정리
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if not CONFIRM_DELETE:
    if connected:
        with driver.session(database=os.environ['NEO4J_DATABASE']) as session:
            result = session.run(
                "MATCH (n) "
                "WHERE size(labels(n)) = 1 AND labels(n)[0] IN $labels "
                "RETURN count(n) AS count",
                labels=node_labels,
            ).single()
        print('삭제 대상 Fp 노드 수:', result['count'])
    else:
        print('미리보기에는 연결이 필요합니다')
elif not connected or (not is_local and not ALLOW_REMOTE):
    print('삭제 차단: 로컬 연결 또는 ALLOW_REMOTE=True 필요')
else:
    with driver.session(database=os.environ['NEO4J_DATABASE']) as session:
        result = session.run(
            "MATCH (n) "
            "WHERE any(label IN labels(n) WHERE label IN $labels) "
            "AND all(label IN labels(n) WHERE label IN $labels) "
            "DETACH DELETE n",
            labels=node_labels,
        )
        result.consume()
    print('Fp 노드 정리 완료')


# 연결 닫기
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if driver is not None:
    driver.close()
    print('연결 닫음')
else:
    print('열린 연결 없음')
